<a href="https://colab.research.google.com/github/IsaacJudeSAcera/GROUP2Rosal_Project_Template_ComputerScience2/blob/main/Rosal_Attendance_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
url = "https://raw.githubusercontent.com/IsaacJudeSAcera/GROUP2Rosal_Project_Template_ComputerScience2/refs/heads/main/data/attendance.json"
response = requests.get(url)
attendance = response.json()
print(attendance)

In [ ]:
!pip install firebase-admin

In [ ]:
import firebase_admin
from firebase_admin import credentials, db
# Load the private key
cred = credentials.Certificate("/content/firebase_key.json")
# Initialize the app with your database URL
firebase_admin.initialize_app(cred, {
 "databaseURL": "https://rosal-attendance-db-61a19-default-rtdb.asia-southeast1.firebasedatabase.app/"
})
print("Firebase connected successfully!")

In [ ]:
import requests
import json
url = "https://raw.githubusercontent.com/IsaacJudeSAcera/GROUP2Rosal_Project_Template_ComputerScience2/refs/heads/main/data/attendance.json"
response = requests.get(url)
attendance = response.json()
attendance_counts = {}
for student in attendance:
    for date_entry in student['dates']:
        if date_entry['status'] == 'Present':
            date = date_entry['date']
            attendance_counts[date] = attendance_counts.get(date, 0) + 1
if attendance_counts:
    max_attendance = max(attendance_counts.values())
    min_attendance = min(attendance_counts.values())
    dates_highest_attendance = [date for date, count in attendance_counts.items() if count == max_attendance]
    dates_lowest_attendance = [date for date, count in attendance_counts.items() if count == min_attendance]
    print(f"Date(s) with highest attendance ({max_attendance} presents): {', '.join(dates_highest_attendance)}")
    print(f"Date(s) with lowest attendance ({min_attendance} presents): {', '.join(dates_lowest_attendance)}")
else:
    print("No attendance data found to calculate highest and lowest attendance.")

In [ ]:
ref = db.reference("attendance")
while True:
    print("\n===== STUDENT DATABASE MENU =====")
    print("1. Display Students")
    print("2. Add Student")
    print("3. Update Student")
    print("4. Delete Student")
    print("5. Features Menu")
    print("6. Exit")
    choice = input("Enter your choice: ")

    if choice == "1":
        attendance_data_from_db = ref.get()
        print("\nStudent List:")
        if attendance_data_from_db:
            processed_students = []
            if isinstance(attendance_data_from_db, dict):
                for student_id, student_details in attendance_data_from_db.items():
                    if 'student_id' not in student_details:
                        student_details['student_id'] = student_id
                    processed_students.append(student_details)
            elif isinstance(attendance_data_from_db, list):
                processed_students = attendance_data_from_db
            else:
                print("Unexpected data format from Firebase.")
                continue

            for student in processed_students:
                date_summaries = []
                if 'dates' in student:
                    for date_entry in student['dates']:
                        date_summaries.append(f"{date_entry.get('date', 'N/A')}: {date_entry.get('status', 'N/A')}")
                all_dates_str = ", ".join(date_summaries)
                print(f"ID: {student.get('student_id', 'N/A')}, Name: {student.get('name', 'N/A')}, Dates: [{all_dates_str}]")
        else:
            print("No students found in the database.")

    elif choice == "2":
        student_id = input("Enter student ID: ")
        name = input("Enter student name: ")
        dates = []
        while True:
            date = input("Enter date (or 'done' to finish): ")
            if date.lower() == 'done':
                break
            status = input("Enter status (Present/Absent/Tardy/Cut Class/Absent - Excused/Absent - Unexcused): ")
            dates.append({"date": date, "status": status})

        new_student = {
            "student_id": student_id,
            "name": name,
            "dates": dates
        }
        ref.child(student_id).set(new_student)
        print(f"Student {name} (ID: {student_id}) added successfully!")

    elif choice == "3":
        student_id_to_update = input("Enter the ID of the student to update: ")
        student_to_update = ref.child(student_id_to_update).get()

        if student_to_update:
            print(f"Current student details: Name: {student_to_update.get('name', 'N/A')}")
            new_name = input("Enter new name (leave blank to keep current): ")
            if new_name:
                ref.child(student_id_to_update).update({'name': new_name})
                print("Name updated.")

            update_dates = input("Do you want to update attendance dates? (yes/no): ").lower()
            if update_dates == 'yes':
                new_dates = []
                print("Enter new dates. Type 'done' when finished with all dates for this student.")
                while True:
                    date = input("Enter date (or 'done' to finish): ")
                    if date.lower() == 'done':
                        break
                    status = input("Enter status (Present/Absent/Tardy/Cut Class/Absent - Excused/Absent - Unexcused): ")
                    new_dates.append({"date": date, "status": status})
                ref.child(student_id_to_update).update({'dates': new_dates})
                print("Attendance dates updated.")
            print(f"Student {student_id_to_update} updated successfully!")
        else:
            print(f"Student with ID {student_id_to_update} not found.")

    elif choice == "4":
        student_id_to_delete = input("Enter the ID of the student to delete: ")
        student_to_delete = ref.child(student_id_to_delete).get()

        if student_to_delete:
            confirm = input(f"Are you sure you want to delete {student_to_delete.get('name', 'N/A')} (ID: {student_id_to_delete})? (yes/no): ").lower()
            if confirm == 'yes':
                ref.child(student_id_to_delete).delete()
                print(f"Student {student_id_to_delete} deleted successfully!")
            else:
                print("Deletion cancelled.")
        else:
            print(f"Student with ID {student_id_to_delete} not found.")

    elif choice == "5":
        print("\n===== ATTENDANCE FEATURES ====")
        attendance_data_from_db = ref.get()
        current_attendance_list = []
        if attendance_data_from_db:
            if isinstance(attendance_data_from_db, dict):
                for student_id, student_details in attendance_data_from_db.items():
                    if 'student_id' not in student_details:
                        student_details['student_id'] = student_id
                    current_attendance_list.append(student_details)
            elif isinstance(attendance_data_from_db, list):
                current_attendance_list = attendance_data_from_db

        if not current_attendance_list:
            print("No attendance data available for features.")
            continue

        # --- Calculations for features ---
        daily_present_counts = {}
        daily_absent_counts = {}
        all_unique_dates = set()
        absence_types_counts = {}
        absence_categories = ['Tardy', 'Cut Class', 'Absent - Excused', 'Absent - Unexcused']

        for student in current_attendance_list:
            if 'dates' in student:
                for date_entry in student['dates']:
                    date = date_entry['date']
                    status = date_entry['status']
                    all_unique_dates.add(date)

                    if status == 'Present':
                        daily_present_counts[date] = daily_present_counts.get(date, 0) + 1
                    else:
                        daily_absent_counts[date] = daily_absent_counts.get(date, 0) + 1
                        if status in absence_categories:
                            absence_types_counts[status] = absence_types_counts.get(status, 0) + 1

        # 1. Average students present per day
        total_present_count = sum(daily_present_counts.values())
        num_dates_with_presence = len(daily_present_counts)
        if num_dates_with_presence > 0:
            avg_present_per_day = total_present_count / num_dates_with_presence
            print(f"1. On average, {avg_present_per_day:.2f} students are present per day.")
        else:
            print("1. No present attendance recorded.")

        # 2. Date(s) with lowest attendance
        if daily_present_counts:
            min_attendance = min(daily_present_counts.values())
            dates_lowest_attendance = [date for date, count in daily_present_counts.items() if count == min_attendance]
            print(f"2. Date(s) with lowest attendance ({min_attendance} presents): {', '.join(dates_lowest_attendance)}")
        else:
            print("2. No present attendance data to determine lowest attendance.")

        # 3. Most common absence type
        if absence_types_counts:
            most_common_absence_type = max(absence_types_counts, key=absence_types_counts.get)
            most_common_absence_count = absence_types_counts[most_common_absence_type]
            print(f"3. Most common absence type: {most_common_absence_type} ({most_common_absence_count} occurrences).")
        else:
            print("3. No absences recorded.")

        # 4. Date(s) with highest attendance
        if daily_present_counts:
            max_attendance = max(daily_present_counts.values())
            dates_highest_attendance = [date for date, count in daily_present_counts.items() if count == max_attendance]
            print(f"4. Date(s) with highest attendance ({max_attendance} presents): {', '.join(dates_highest_attendance)}")
        else:
            print("4. No present attendance data to determine highest attendance.")

        # 5. Average students absent per day
        total_absent_count = sum(daily_absent_counts.values())
        num_overall_unique_dates = len(all_unique_dates)
        if num_overall_unique_dates > 0:
            avg_absent_per_day = total_absent_count / num_overall_unique_dates
            print(f"5. On average, {avg_absent_per_day:.2f} students are absent per day.")
        else:
            print("5. No attendance data to determine average absences.")

    elif choice == "6":
        break
    else:
        print("Invalid choice. Please try again.")


===== STUDENT DATABASE MENU =====
1. Display Students
2. Add Student
3. Update Student
4. Delete Student
5. Features Menu
6. Exit

Student List:
ID: S001, Name: Alice Johnson, Dates: [2025-09-01: Present, 2025-09-02: Absent - Excused, 2025-09-03: Present, 2025-09-04: Tardy, 2025-09-05: Cut Class, 2025-09-06: Present]
ID: S002, Name: Brian Smith, Dates: [2025-09-01: Tardy, 2025-09-02: Present, 2025-09-03: Absent - Unexcused, 2025-09-04: Present, 2025-09-05: Present, 2025-09-06: Cut Class]
ID: S003, Name: Carla Gomez, Dates: [2025-09-01: Present, 2025-09-02: Present, 2025-09-03: Absent - Excused, 2025-09-04: Present, 2025-09-05: Tardy, 2025-09-06: Absent - Unexcused]
ID: S004, Name: Daniel Lee, Dates: [2025-09-01: Cut Class, 2025-09-02: Present, 2025-09-03: Present, 2025-09-04: Tardy, 2025-09-05: Absent - Unexcused, 2025-09-06: Present]

===== STUDENT DATABASE MENU =====
1. Display Students
2. Add Student
3. Update Student
4. Delete Student
5. Features Menu
6. Exit
Student Jane Felisan 